<a href="https://colab.research.google.com/github/fserracrespi/ir2_Project/blob/mhm-dev/pairs/10-gpl/reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPL (Wang et al., NAACL 2022) on the IR 2 testbed, scaled down

This notebook implements the Generative Pseudo Labeling (GPL) pipeline for unsupervised domain adaptation of a dense retriever, on a single Colab GPU. It can train **any sentence-transformers student** on any of the three testbed corpora (SciFact, NFCorpus, ArguAna), and compares it with zero-shot and GenQ using the testbed's own evaluation. For the presentation, we used:
- sentence-transformers/all-MiniLM-L6-v2
- BAAI/bge-small-en-v1.5


## The pipeline

1. **Query generation:** T5 (doc2query) writes `QUERIES_PER_DOC` queries for each document in a random part of the corpus (`TRAIN_FRACTION`).
2. **Negative mining:** for every generated query, `N_MINE` negatives from each of the paper's two dense retrievers (pooled), or `2 * N_MINE` from BM25 for the ablation. The query's own positive is removed.
3. **Pseudo labeling:** one random negative per query; the cross-encoder gives the margin CE(q, pos) − CE(q, neg).
4. **Training:** the student learns the margins with MarginMSE. GenQ (same queries, binary labels, MNRL) is the baseline.

## How to use it

- Set the three variables at the top of the Config cell: `DATASET`, `RETRIEVAL_TYPE` (`"dense"` or `"BM25"`) and `STUDENT_MODEL`, then *Run all*. The per-corpus settings (`TRAIN_FRACTION`, `QUERIES_PER_DOC`) follow `DATASET` automatically.
- For a **normalized (cosine) student** such as BGE-small or MiniLM, set `RESCALE = (-2.0, 2.0)` (the GPL repository's range for cosine students). For an **unnormalized dot-product student** such as the paper's, set `RESCALE = None` (raw margins, as in the paper).
- Every slow step saves its result to `DRIVE` under a name that contains every setting it depends on, and loads it instead of recomputing when it already exists. Dense and BM25 runs share the same generated queries, and GenQ only trains once per dataset and student.
- No test query and no relevance label is used for training: `queries` and `qrels` only appear inside `evaluate`.

---

# Reproduction of a key result at small scale

**Goal:** reproduce the paper's central claim, GPL > QGen > zero-shot on SciFact (paper, Table 1: 0.571 → 0.638 → 0.664 nDCG@10), at small scale, using the paper's own starting model and label setup.

**To rerun it:** `DATASET = "scifact"`, `RETRIEVAL_TYPE = "dense"`, `STUDENT_MODEL = "GPL/msmarco-distilbert-margin-mse"`, `RESCALE = None`.

### Setup

- **Student:** `GPL/msmarco-distilbert-margin-mse`, the paper's zero-shot checkpoint (DistilBERT trained on MS MARCO, mean pooling, unnormalized dot product).
- **Labels:** raw cross-encoder margins, no rescaling, as in the paper. Range −17.3 to +21.5, median 12.9; 2% negative.
- **Data:** identical

## 0. Setup

In [1]:
# clones our branch on Colab; before the pull request this may point to Kim's repository instead
import os
if not os.path.exists("testbed.py"):
    if not os.path.exists("/content/ir2_Project"):
        !git clone -q -b mhm-dev https://github.com/fserracrespi/ir2_Project /content/ir2_Project
    %cd /content/ir2_Project
    !git pull -q
!pip install -q -r requirements.txt datasets accelerate peft
!git branch --show-current

/content/ir2_Project
mhm-dev


## 1. Config

Change these three for each run to obtain different models:

In [2]:
DATASET = "scifact"                       # "scifact" | "nfcorpus" | "arguana"
RETRIEVAL_TYPE = "dense"                  # "dense" (paper) | "BM25" (ablation)
STUDENT_MODEL = "GPL/msmarco-distilbert-margin-mse"  # for class, we used sentence-transformers/all-MiniLM-L6-v2 and BAAI/bge-small-en-v1.5

Fixed for all runs, so they stay comparable. Paper values in the comments.

In [3]:
import random, json
import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import (CrossEncoder, SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses, util)
from sentence_transformers.training_args import BatchSamplers
import testbed

SEED = 0
FULL_TEXT = False          # True: full title + abstract, cut at MAX_SEQ_LEN tokens (paper); False: testbed's 200 words
# per-corpus settings: about 7,800 generated queries per corpus, at least 3 queries per passage (paper's minimum)
CORPUS_SETTINGS = {"scifact":  {"TRAIN_FRACTION": 1/2, "QUERIES_PER_DOC": 3},
                   "nfcorpus": {"TRAIN_FRACTION": 1/2, "QUERIES_PER_DOC": 4},
                   "arguana":  {"TRAIN_FRACTION": 1/3, "QUERIES_PER_DOC": 3}}
TRAIN_FRACTION = CORPUS_SETTINGS[DATASET]["TRAIN_FRACTION"]    # share of the corpus that gets generated queries
QUERIES_PER_DOC = CORPUS_SETTINGS[DATASET]["QUERIES_PER_DOC"]
print(f"{DATASET}: TRAIN_FRACTION = {TRAIN_FRACTION:.3g}, QUERIES_PER_DOC = {QUERIES_PER_DOC}")

N_MINE = 50               # negatives per dense retriever (paper: 50); BM25 keeps 2 * N_MINE so pool sizes match
N_NEGS_QUERY = 1          # training examples per generated query, each with its own random negative (paper: 1 per example)
RESCALE = None # (-2.0, 2.0)     # GPL repo's rescale_range for cosine students; None = raw cross-encoder margins (paper, dot product)
MAX_SEQ_LEN = 350         # paper
BATCH = 32                # GPL batch size (paper)
GENQ_BATCH = 75           # QGen batch size (paper); lower it if the GPU runs out of memory
LR = 2e-5
USE_FP16 = True           # mixed precision (the GPL repo's use_amp)
USE_LORA = False          # optional extra: GPL with a LoRA adapter instead of full fine-tuning
RETRAIN = False           # False: if a run file already exists, load and score it instead of training again

GEN_MODEL = "doc2query/msmarco-t5-base-v1"
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
DENSE_MINERS = ["sentence-transformers/msmarco-distilbert-base-v3",
                "sentence-transformers/msmarco-MiniLM-L6-v3"]
#DRIVE = "/content/drive/MyDrive/gpl"
DRIVE = "/content/gpl"

scifact: TRAIN_FRACTION = 0.5, QUERIES_PER_DOC = 3


/tmp/ipykernel_792/1249253987.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (CrossEncoder, SentenceTransformer, SentenceTransformerTrainer,
/tmp/ipykernel_792/1249253987.py:7: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


Derived settings and file names. Each name contains every setting its step depends on, so a changed setting never loads data made with another one.

In [4]:
STUD_SHORT = ("bge" if "bge" in STUDENT_MODEL.lower()
              else "minilm" if "minilm" in STUDENT_MODEL.lower()
              else STUDENT_MODEL.split("/")[-1])
QUERY_PREFIX = testbed.BGE_QUERY if STUD_SHORT == "bge" else ""   # MiniLM was trained without a prefix
assert RETRIEVAL_TYPE in ("dense", "BM25"), RETRIEVAL_TYPE

text_tag = "full" if FULL_TEXT else "w200"
miner_tag = "dense" if RETRIEVAL_TYPE == "dense" else "bm25stem"
rescale_tag = "raw" if RESCALE is None else f"rescale{RESCALE[1]:g}"

GEN_TAG = f"{DATASET}_{text_tag}_frac{TRAIN_FRACTION}_qpp{QUERIES_PER_DOC}_seed{SEED}"   # queries: shared by dense, BM25, GenQ
MINE_TAG = f"{GEN_TAG}_{miner_tag}{N_MINE}"                                                # pools
DATA_TAG = f"{MINE_TAG}_neg{N_NEGS_QUERY}"                                                 # pseudo-labelled examples
# run folders hold one file per dataset (<folder>/<dataset>.json), the layout testbed.py score expects
GPL_FOLDER = (f"gpl_{STUD_SHORT}_{miner_tag}{N_MINE}_{text_tag}_frac{TRAIN_FRACTION}_qpp{QUERIES_PER_DOC}"
              f"_neg{N_NEGS_QUERY}_{rescale_tag}_lr{LR:g}_seed{SEED}")
GENQ_FOLDER = f"genq_{STUD_SHORT}_{text_tag}_frac{TRAIN_FRACTION}_qpp{QUERIES_PER_DOC}_bs{GENQ_BATCH}_lr{LR:g}_seed{SEED}"
ZERO_FOLDER = f"zeroshot_{STUD_SHORT}_{text_tag}"

PATHS = {
    "queries":  f"{DRIVE}/queries/{GEN_TAG}.json",
    "pools":    f"{DRIVE}/pools/{MINE_TAG}.json",
    "examples": f"{DRIVE}/examples/{DATA_TAG}.json",
    "zero_run": f"{DRIVE}/runs/{ZERO_FOLDER}/{DATASET}.json",
    "genq_run": f"{DRIVE}/runs/{GENQ_FOLDER}/{DATASET}.json",
    "gpl_run":  f"{DRIVE}/runs/{GPL_FOLDER}/{DATASET}.json",
    "results":  f"{DRIVE}/results/{GPL_FOLDER}_{DATASET}.json",
}
for k, v in PATHS.items():
    print(f"{k:9} {v}")

queries   /content/gpl/queries/scifact_w200_frac0.5_qpp3_seed0.json
pools     /content/gpl/pools/scifact_w200_frac0.5_qpp3_seed0_dense50.json
examples  /content/gpl/examples/scifact_w200_frac0.5_qpp3_seed0_dense50_neg1.json
zero_run  /content/gpl/runs/zeroshot_msmarco-distilbert-margin-mse_w200/scifact.json
genq_run  /content/gpl/runs/genq_msmarco-distilbert-margin-mse_w200_frac0.5_qpp3_bs75_lr2e-05_seed0/scifact.json
gpl_run   /content/gpl/runs/gpl_msmarco-distilbert-margin-mse_dense50_w200_frac0.5_qpp3_neg1_raw_lr2e-05_seed0/scifact.json
results   /content/gpl/results/gpl_msmarco-distilbert-margin-mse_dense50_w200_frac0.5_qpp3_neg1_raw_lr2e-05_seed0_scifact.json


In [5]:
from google.colab import drive
#drive.mount("/content/drive")
for p in PATHS.values():
    os.makedirs(os.path.dirname(p), exist_ok=True)

def cached(path, compute):
    # load `path` if it exists, otherwise compute, save and return the result
    if os.path.exists(path):
        print("loaded", path)
        return json.load(open(path))
    result = compute()
    json.dump(result, open(path, "w"))
    print("saved", path)
    return result

def free_gpu():
    import gc; gc.collect(); torch.cuda.empty_cache()

print(testbed.device())

cuda


## 2. Data and zero-shot baseline

`evaluate` is the testbed's search and scoring, on the same document text the pipeline trains on (`all_texts`) and with the student's own query prefix.

In [6]:
corpus, queries, qrels = testbed.load(DATASET)
doc_ids = list(corpus)
idx = {d: n for n, d in enumerate(doc_ids)}
# title + abstract for all models (paper), in corpus order, aligned with doc_ids
all_texts = ([corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids] if FULL_TEXT
             else testbed.doc_texts(corpus))
N_TRAIN = int(len(doc_ids) * TRAIN_FRACTION)
print(len(corpus), "documents,", len(queries), "test queries,", N_TRAIN, "documents get generated queries")

downloading scifact


5183 documents, 300 test queries, 2591 documents get generated queries


In [7]:
def load_student():
    m = SentenceTransformer(STUDENT_MODEL, device=testbed.device())
    m.max_seq_length = MAX_SEQ_LEN
    return m

def evaluate(model, label, run_path=None):
    run = testbed.dense_search(model, corpus, queries, texts=all_texts, query_prefix=QUERY_PREFIX)
    if run_path:
        testbed.save_run(run, run_path)
    return score(run, label)

def score(run, label):
    _, ndcg, recall = testbed.score_run(run, DATASET)
    print(f"{label:28} nDCG@10 {ndcg.mean():.3f}  Recall@100 {recall.mean():.3f}")
    return ndcg

model = load_student()
before = evaluate(model, f"{STUD_SHORT} zero-shot", PATHS["zero_run"])
del model; free_gpu()

config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  265MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

msmarco-distilbert-margin-mse zero-shot nDCG@10 0.519  Recall@100 0.830


# 2 GPL training of the student, step by step
## 2.1 T5 query generation

For each document in the training half, `QUERIES_PER_DOC` queries, with the paper's sampling (top-k 25, top-p 0.95, temperature 1.0).

In [8]:
# the documents that get queries: a seeded random half of the corpus
shuffled = doc_ids[:]
random.Random(SEED).shuffle(shuffled)
train_ids = shuffled[:N_TRAIN]
title_abstract_train = [all_texts[idx[d]] for d in train_ids]

def generate_queries():
    from transformers import T5Tokenizer, T5ForConditionalGeneration
    t5_tokenizer = T5Tokenizer.from_pretrained(GEN_MODEL)
    t5_model = T5ForConditionalGeneration.from_pretrained(GEN_MODEL).to(testbed.device())
    torch.manual_seed(SEED)
    generated = []
    for i in range(0, len(title_abstract_train), BATCH):
        ids = train_ids[i: i + BATCH]
        texts = title_abstract_train[i: i + BATCH]
        encoded_out = t5_tokenizer(texts, max_length=MAX_SEQ_LEN, truncation=True, padding=True, return_tensors="pt").to(t5_model.device)
        generation_out = t5_model.generate(**encoded_out, max_length=64, do_sample=True, top_k=25, top_p=0.95,
                                           num_return_sequences=QUERIES_PER_DOC)
        decoded = t5_tokenizer.batch_decode(generation_out, skip_special_tokens=True)
        generated += [{"query": q, "pos": ids[k // QUERIES_PER_DOC]} for k, q in enumerate(decoded)]
    del t5_model; free_gpu()
    return generated

generated = cached(PATHS["queries"], generate_queries)

# checks: right dataset, right size, right documents
assert all(g["pos"] in corpus for g in generated), "queries belong to another dataset"
assert len(generated) == N_TRAIN * QUERIES_PER_DOC, (len(generated), N_TRAIN * QUERIES_PER_DOC)
assert {g["pos"] for g in generated} == set(train_ids), "saved queries come from other training documents"
print(len(generated), "queries")
for g in generated[:3]:
    print(corpus[g["pos"]]["title"][:90], "\n  ->", g["query"])

loaded /content/gpl/queries/scifact_w200_frac0.5_qpp3_seed0.json
7773 queries
Cardiovascular and renal benefits of dry bean and soybean intake. 
  -> benefits of eating soybeans and soy
Cardiovascular and renal benefits of dry bean and soybean intake. 
  -> benefits of eating soy
Cardiovascular and renal benefits of dry bean and soybean intake. 
  -> does dry soybeans contain any vitamin or minerals


Example queries for NFCorpus

corpus title: 'Towards a new system of health: the challenge of Western disease.'
- query: how to change the climate in western disease
- query: which is the most important issue for a new system of health?
- query: does government support health education

Example queries for SciFact (3 queries / passage)

corpus title: 'A temporal shift in the circuits mediating retrieval of fear memory'
- query 1: how do conditioned nerves retrieve fear
- query 2: which neuron is responsible for fear retrieval
- query 3: does the pvt stimulate fear

Other SciFact examples:
- 'Cardiovascular and renal benefits of dry bean and soybean intake.' → is soy good for the kidneys
- 'The 5TMM series: a useful in vivo mouse model of human multiple myeloma.' (abstract is actually a patent for a sink) → what is a combination sink and dishwashing apparatus
- 'Transient receptor potential channels as drug targets…' → what is the function of the trp gene

## 2.2 Negative mining

**Dense (paper):** two dense retrievers from Sentence-Transformers, `msmarco-distilbert-base-v3` and `msmarco-MiniLM-L-6-v3`, `N_MINE` negatives each (paper: 50), pooled without duplicates.

**BM25 (ablation, paper Table 6):** `2 * N_MINE` negatives so the pool size matches, with the same preprocessing as `testbed.bm25_search` (English stopwords removed, Porter stemming, k1 = 1.2, b = 0.75), closer to the Elasticsearch BM25 the GPL repository uses.

In both cases the query's own positive is removed, and the whole corpus is searched.

In [9]:
def mine_dense(model_name):
    miner = SentenceTransformer(model_name, device=testbed.device())
    # normalized embeddings: cosine similarity, as these retrievers were trained
    D = miner.encode(all_texts, normalize_embeddings=True, batch_size=BATCH, convert_to_tensor=True, show_progress_bar=True)
    Q = miner.encode([g["query"] for g in generated], normalize_embeddings=True, batch_size=BATCH, convert_to_tensor=True,
                     show_progress_bar=True)
    hits = util.semantic_search(Q, D, top_k=N_MINE + 1)    # one extra, since the positive is removed
    del miner; free_gpu()
    return [[doc_ids[h["corpus_id"]] for h in hs if doc_ids[h["corpus_id"]] != g["pos"]][:N_MINE]
            for g, hs in zip(generated, hits)]

def mine_bm25():
    # same preprocessing as testbed.bm25_search (stopwords + Porter stemming)
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    from rank_bm25 import BM25Okapi
    from sklearn.feature_extraction.text import CountVectorizer
    nltk.download("stopwords", quiet=True)
    sw, porter, analyze = set(stopwords.words("english")), PorterStemmer(), CountVectorizer().build_analyzer()
    def tokenize(t): return [porter.stem(x) for x in analyze(t) if x not in sw]
    bm25 = BM25Okapi([tokenize(t) for t in all_texts], k1=1.2, b=0.75)
    pools = []
    for g in generated:
        top_docs = np.argsort(-bm25.get_scores(tokenize(g["query"])))[:N_MINE * 2 + 1]
        pools.append([doc_ids[j] for j in top_docs if doc_ids[j] != g["pos"]][:N_MINE * 2])
    return pools

def mine():
    if RETRIEVAL_TYPE == "BM25":
        return mine_bm25()
    neg_distil, neg_minilm = (mine_dense(name) for name in DENSE_MINERS)
    # pool the two miners' results, keeping each document once
    return [list(dict.fromkeys(a + b)) for a, b in zip(neg_distil, neg_minilm)]

pool_ids = cached(PATHS["pools"], mine)

sizes = [len(p) for p in pool_ids]
assert len(pool_ids) == len(generated)
assert all(g["pos"] not in p for g, p in zip(generated, pool_ids)), "a positive is in its own pool"
print(len(pool_ids), "pools, size min/mean/max:", min(sizes), round(float(np.mean(sizes)), 1), max(sizes))

loaded /content/gpl/pools/scifact_w200_frac0.5_qpp3_seed0_dense50.json
7773 pools, size min/mean/max: 60 88.0 100


In [10]:
# look at one query and its first negatives
i = 9
print("QUERY:   ", generated[i]["query"])
print("POSITIVE:", corpus[generated[i]["pos"]]["title"])
for d in pool_ids[i][:4]:
    print("  negative:", corpus[d]["title"])

QUERY:    what is the genotype for apolipoprotein e
POSITIVE: Apolipoprotein E genotype and concussion in college athletes.
  negative: Apolipoprotein E and atherosclerosis: beyond lipid effect.
  negative: Apolipoprotein E: high-avidity binding to beta-amyloid and increased frequency of type 4 allele in late-onset familial Alzheimer disease.
  negative: Apolipoprotein E4 as a predictor of outcomes in pediatric mild traumatic brain injury.
  negative: Disclosure of APOE genotype for risk of Alzheimer's disease.


With dense mining, both miners retrieve different passages for the same query, but both usually rank the positive first, a sign that they understand the generated queries. Having two miners gives more varied negatives, since they encode the texts differently.

## 2.3 Cross-encoder pseudo labeling

The paper trains 140K steps with batch size 32, using each query about 18 times with a fresh negative. We use each query `N_NEGS_QUERY` times (default once): for each, a different negative drawn at random from its pool, and the raw-logit margin CE(q, pos) − CE(q, neg).

In [11]:
def pseudo_label():
    ce = CrossEncoder(CE_MODEL, device=testbed.device())
    rng = random.Random(SEED)
    neg_ids = [rng.sample(p, N_NEGS_QUERY) for p in pool_ids]        # different negatives per query, seeded
    query_texts = [g["query"] for g in generated]
    posi_texts = [all_texts[idx[g["pos"]]] for g in generated]
    posi_ce_scores = ce.predict(list(zip(query_texts, posi_texts)), batch_size=BATCH, show_progress_bar=True)
    examples = []                                                      # [query, pos_id, neg_id, margin]
    for k in range(N_NEGS_QUERY):
        negs_k = [n[k] for n in neg_ids]
        neg_ce_scores = ce.predict(list(zip(query_texts, [all_texts[idx[n]] for n in negs_k])),
                                   batch_size=BATCH, show_progress_bar=True)
        for j, g in enumerate(generated):
            examples.append([g["query"], g["pos"], negs_k[j], float(posi_ce_scores[j] - neg_ce_scores[j])])
    del ce; free_gpu()
    return examples

examples = cached(PATHS["examples"], pseudo_label)
margins = np.array([e[3] for e in examples])
assert len(examples) == len(generated) * N_NEGS_QUERY
print(len(examples), "examples")
print(f"margins: min {margins.min():.1f}  median {np.median(margins):.1f}  max {margins.max():.1f}  below 0: {(margins < 0).mean():.0%}")

loaded /content/gpl/examples/scifact_w200_frac0.5_qpp3_seed0_dense50_neg1.json
7773 examples
margins: min -17.3  median 12.9  max 21.5  below 0: 2%


Most positives get a high score, so the cross-encoder agrees the source passage answers its generated query. Some get a negative score, which can come from badly generated queries or from documents whose abstract does not match the title. Most negatives get a low score, but some score high: likely false negatives. Training on these soft labels instead of QGen's binary 1/0 keeps false negatives close and gives bad queries little weight.

### Rescaling

BGE and MiniLM normalize their embeddings, so their margin is a difference of cosines and lies in [−2, 2], while the raw cross-encoder margins reach about ±20. `rescale_fn` is copied unchanged from the GPL repository: positive and negative margins are each stretched linearly onto their half of `RESCALE`, so 0 stays 0 and no sign flips. With `RESCALE = None` the raw margins are used, as in the paper's dot-product setup.

In [12]:
def rescale_fn(margin, org_min, org_max, new_min, new_max):
    # unchanged from the GPL repository (gpl/toolkit)
    if org_max == 0 or org_min == 0:
        pass
    elif margin > 0:
        org_min, new_min = 0, 0
    else:
        assert margin <= 0
        org_max, new_max = 0, 0
    return new_min + (margin - org_min) * (new_max - new_min) / (org_max - org_min)

if RESCALE is None:
    labels = margins.tolist()
else:
    org_min, org_max = float(margins.min()), float(margins.max())
    labels = [rescale_fn(m, org_min, org_max, *RESCALE) for m in margins]
print(f"labels: min {min(labels):.2f}  max {max(labels):.2f}")

labels: min -17.26  max 21.47


## 2.4 Training

Both runs start from the same downloaded student, with the same generated queries.

In [13]:
def train(model, data, loss, output_dir, lr=LR, batch=BATCH, no_duplicates=False):
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir, num_train_epochs=1, per_device_train_batch_size=batch,
        learning_rate=lr, logging_steps=20, save_strategy="no", report_to="none", seed=SEED,
        fp16=USE_FP16 and torch.cuda.is_available(), use_cpu=not torch.cuda.is_available(),
        batch_sampler=BatchSamplers.NO_DUPLICATES if no_duplicates else BatchSamplers.BATCH_SAMPLER)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=loss).train()

def trained_or_cached(run_path, label, build_and_train):
    # train and evaluate, or score the saved run if it exists and RETRAIN is False
    if os.path.exists(run_path) and not RETRAIN:
        print("loaded", run_path)
        return score(json.load(open(run_path)), label)
    model = build_and_train()
    ndcg = evaluate(model, label, run_path)
    del model; free_gpu()
    return ndcg

### GPL: MarginMSE on the pseudo-labels

In [14]:
def train_gpl():
    gpl_data = Dataset.from_dict({
        "query":    [QUERY_PREFIX + e[0] for e in examples],
        "positive": [all_texts[idx[e[1]]] for e in examples],
        "negative": [all_texts[idx[e[2]]] for e in examples],
        "label":    labels,
    })
    print(len(gpl_data), "examples,", len(gpl_data) // BATCH, "steps")
    student = load_student()
    train(student, gpl_data, losses.MarginMSELoss(student), "runs/gpl")
    # student.save(f"{DRIVE}/models/{GPL_FOLDER}/{DATASET}") uncomment to save the checkpoint
    return student

after_gpl = trained_or_cached(PATHS["gpl_run"], f"GPL ({RETRIEVAL_TYPE} negatives)", train_gpl)

7773 examples, 242 steps


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
20,21.995853
40,19.038776
60,17.368620
80,19.120346
100,18.809341
120,19.865157
140,19.353214
160,16.720490
180,19.418001
200,16.062926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GPL (dense negatives)        nDCG@10 0.581  Recall@100 0.865


### GenQ baseline: same queries, binary labels, MNRL with in-batch negatives

As the paper's QGen: cosine similarity, batch size 75, one epoch. `NO_DUPLICATES` keeps the queries of one document out of the same batch, otherwise that document would be an in-batch negative for its own queries. GenQ does not use mined negatives, so it is trained once per dataset and student and reused by the BM25 run.

In [15]:
def train_genq():
    genq_data = Dataset.from_dict({
        "anchor":   [QUERY_PREFIX + g["query"] for g in generated],
        "positive": [all_texts[idx[g["pos"]]] for g in generated],
    })
    print(len(genq_data), "pairs,", len(genq_data) // GENQ_BATCH, "steps")
    genq_model = load_student()
    train(genq_model, genq_data, losses.MultipleNegativesRankingLoss(genq_model), "runs/genq",
          batch=GENQ_BATCH, no_duplicates=True)
    # genq_model.save(f"{DRIVE}/models/{GENQ_FOLDER}/{DATASET}") uncomment to save the checkpoint
    return genq_model

after_genq = trained_or_cached(PATHS["genq_run"], "GenQ", train_genq)


7773 pairs, 103 steps


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
20,1.053036
40,0.518515
60,0.411471
80,0.371526
100,0.373151


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GenQ                         nDCG@10 0.577  Recall@100 0.882


## 3. Did it change anything?

Here we compare the results of:

1. Zero-shot
2. Training the student with GPL
3. Training the student with GenQ

The same test as in the results table: a paired t-test and a bootstrap interval over the test queries. A small difference with a large p-value is noise, not a win.

In [16]:
rows = {"zero-shot": before, "GenQ": after_genq, f"GPL ({RETRIEVAL_TYPE})": after_gpl}
comparisons = {
    "GenQ vs zero-shot": testbed.compare(after_genq, before),
    "GPL vs zero-shot":  testbed.compare(after_gpl, before),
    "GPL vs GenQ":       testbed.compare(after_gpl, after_genq),
}
print(f"{DATASET}, student {STUD_SHORT}, {RETRIEVAL_TYPE} negatives")
for name, nd in rows.items():
    print(f"  {name:18} nDCG@10 {nd.mean():.3f}")
for name, c in comparisons.items():
    print(f"  {name:18} {c}")

config = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (str, int, float, bool, list, tuple, type(None)))}
json.dump({"config": config, "ndcg10": {k: float(v.mean()) for k, v in rows.items()}, "comparisons": comparisons},
          open(PATHS["results"], "w"), indent=1)
print("saved", PATHS["results"])

scifact, student msmarco-distilbert-margin-mse, dense negatives
  zero-shot          nDCG@10 0.519
  GenQ               nDCG@10 0.577
  GPL (dense)        nDCG@10 0.581
  GenQ vs zero-shot  {'delta': 0.0578, 'p': 3.1588577942831826e-05, 'ci95': [0.0319, 0.0852]}
  GPL vs zero-shot   {'delta': 0.0625, 'p': 4.3846250724945105e-09, 'ci95': [0.0431, 0.0832]}
  GPL vs GenQ        {'delta': 0.0046, 'p': 0.6882367083110317, 'ci95': [-0.0184, 0.0275]}
saved /content/gpl/results/gpl_msmarco-distilbert-margin-mse_dense50_w200_frac0.5_qpp3_neg1_raw_lr2e-05_seed0_scifact.json


## 4. Optional: LoRA instead of full fine-tuning

Set `USE_LORA = True` in the config. Instead of updating all the weights, a small adapter is trained on top of frozen ones, on the same GPL data. LoRA needs a higher learning rate. `peft` may need `torchao` removed first.

In [17]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

if USE_LORA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    from peft import LoraConfig, TaskType
    lora_model = load_student()
    lora_model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
                                      r=8, lora_alpha=16, lora_dropout=0.1))
    count_params(lora_model)
    lora_data = Dataset.from_dict({
        "query":    [QUERY_PREFIX + e[0] for e in examples],
        "positive": [all_texts[idx[e[1]]] for e in examples],
        "negative": [all_texts[idx[e[2]]] for e in examples],
        "label":    labels,
    })
    train(lora_model, lora_data, losses.MarginMSELoss(lora_model), "runs/gpl_lora", lr=5e-4)
    lora = evaluate(lora_model, "GPL with LoRA", PATHS["gpl_run"].replace("/gpl_", "/gpl_lora_"))
    print("LoRA vs zero-shot:", testbed.compare(lora, before))

## From here to the testbed row

Each run folder on Drive holds one `<dataset>.json` per dataset, the layout `testbed.py score` expects. For the class table, use runs made with `FULL_TEXT = False` (the testbed's 200-word documents, like every baseline):

`python testbed.py score <run folder> --method "GPL + BGE-small" --pair 10-gpl --against bge --out <run folder>/results.json`